In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import polars as pl
import plotly.express as px
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from utilsforecast.losses import *
from utilsforecast.evaluation import evaluate


from utilsforecast.losses import *

from utilsforecast.losses import *

import plotly.io as pio

from utilsforecast.losses import *
from utilsforecast.feature_engineering import fourier, pipeline
from functools import partial
from sklearn.linear_model import RidgeCV
from utilsforecast.losses import *
from plotting_utils import (
    plot_data_availability_heatmap,
    plot_missing_percentage,
)
from statsforecast.models import SklearnModel

from statsforecast.models import (
    SeasonalNaive,
    AutoETS,
    MSTL,
)
from xgboost import XGBRegressor

In [ ]:
pio.templates.default = "plotly_white"

# Introduction to Global Forecasting in Time Series

Forecasting is the art and science of predicting future values based on historical data. Traditionally, time series forecasting has focused on **local models**, where a separate model is trained for each individual time series (for example, one model per household's energy consumption).

However, as datasets grow larger and more complex—often containing thousands or even millions of related time series—a new approach has emerged: **global forecasting**.

---

## What is Global Forecasting?

**Global forecasting** refers to building a single model that learns from *all* available time series simultaneously, rather than modeling each series in isolation.

- **Local Model:** One model per series (e.g., one model for each household).
- **Global Model:** One model for all series, leveraging shared patterns across them.

---

## Why Use Global Models?

- **Leverage Shared Patterns:** Many time series in a dataset may exhibit similar behaviors (e.g., daily or weekly cycles in energy usage). A global model can learn these commonalities, improving accuracy—especially for series with limited data.
- **Scalability:** Training and maintaining thousands of local models can be computationally expensive and hard to manage. A global model is more efficient and easier to deploy.
- **Robustness:** By pooling information, global models can generalize better and are less sensitive to noise or missing data in individual series.

---

## Real-World Analogy

Imagine teaching a group of students to solve math problems. If you tutor each student separately (local modeling), you might miss common mistakes or strategies that could help everyone. By teaching the whole class together (global modeling), you can address shared challenges and help everyone learn more efficiently.

---

## How Do Global Models Work?

Global models treat the dataset as a collection of related time series. They use features that identify each series (like a household ID) and can incorporate additional information (such as time, weather, or holidays) to make predictions.

- **Machine Learning Models:** Algorithms like Ridge Regression, XGBoost, or neural networks can be trained globally, using all series together.
- **Feature Engineering:** Global models often rely on engineered features (lags, rolling statistics, seasonality indicators) to capture both individual and shared patterns.

---

## Mathematical Perspective

Suppose we have $N$ time series, each with observations $y_{i, t}$ for series $i$ at time $t$.  
A global model learns a function $f$ such that:

$$
\hat{y}_{i, t+h} = f(\text{features}_{i, t}, \theta)
$$

where $\text{features}_{i, t}$ include past values, time indicators, and possibly series identifiers, and $\theta$ are the parameters learned from *all* series.

---

## When Are Global Models Most Effective?

- When time series are related or share similar seasonal patterns.
- When some series have limited historical data.
- When computational efficiency and scalability are important.

---

In the next sections, we'll explore how to implement global forecasting using the `mlforecast` library, engineer powerful features, and compare the results to our baseline local models. This will help us understand the strengths and limitations of global approaches in real-world forecasting tasks.

In [ ]:
data = pl.read_parquet(
    [
        "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet",
    ]
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)

In [ ]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [ ]:
data = data.select(
    [
        time_,
        id_,
        target_,
        "Acorn",
        "Acorn_grouped",
        "holidays",
        "visibility",
        "windBearing",
        "temperature",
        "dewPoint",
        "pressure",
        "apparentTemperature",
        "windSpeed",
        "precipType",
        "icon",
        "humidity",
        "summary",
    ]
).explode(
    [
        time_,
        target_,
        "holidays",
        "visibility",
        "windBearing",
        "temperature",
        "dewPoint",
        "pressure",
        "apparentTemperature",
        "windSpeed",
        "precipType",
        "icon",
        "humidity",
        "summary",
    ]
)
data.head()

In [ ]:
print("Number of households", data.get_column("unique_id").n_unique())

In [ ]:
data = data.filter(
    pl.col(target_).is_not_null().sum().truediv(pl.len()).over(id_).ge(0.8)
)
print("Number of households", data.get_column("unique_id").n_unique())

In [ ]:
# ## Filtering Series with a Full Week of Recent Data (No Missing Values in Last 336 Steps)

# For half-hourly data, one week = 7 days * 24 hours * 2 = 336 periods.
validation_window = 336

# Step 1: Sort data by unique_id and timestamp to ensure correct ordering
data_sorted = data.sort([id_, time_])

# Step 2: For each series, check if the last 336 target values are all present (not null)
# We'll use polars' group_by and tail to efficiently select the last 336 rows per series
last_week = (
    data_sorted.group_by(id_, maintain_order=True)
    .tail(validation_window)
    .with_columns(
        [
            # Mark if target is not null
            pl.col(target_).is_not_null().alias("not_null")
        ]
    )
)

# Step 3: For each series, check if all last 336 are not null
valid_series = (
    last_week.group_by(id_)
    .agg(pl.col("not_null").all().alias("full_week"))
    .filter(pl.col("full_week"))
    .get_column(id_)
    .to_list()
)

# Step 4: Filter the main data to keep only these valid series
data = data.filter(pl.col(id_).is_in(valid_series))

print(
    f"Number of households with a complete last week: {data.get_column(id_).n_unique()}"
)

In [ ]:
import random

# ## Selecting a Random Sample of 100 Households for Local Forecasting

# To ensure our analysis is representative and unbiased, we'll randomly select 100 unique households
# from the list of valid series. This helps avoid any unintended patterns that might arise from
# simply taking the first 100 households in the list.


# Set a random seed for reproducibility (so results are consistent each time you run the code)
random.seed(42)

# Randomly sample 100 unique household IDs from the valid_series list
selected_ids = random.sample(valid_series, 100)

# Filter the main data to include only these 100 households
data = data.filter(pl.col(id_).is_in(selected_ids))

print(f"Number of households selected: {data.get_column(id_).n_unique()}")

In [ ]:
id_orders = data.select(pl.col(id_)).unique()

In [ ]:
plot_missing_percentage(data, id_orders=id_orders)

In [ ]:
plot_data_availability_heatmap(data, id_orders=id_orders)

In [ ]:
data = data.sort([id_, time_]).with_columns(
    target_col.fill_null(target_col.shift(48 * 7).over(id_))
)

In [ ]:
plot_missing_percentage(data, id_orders=id_orders)

In [ ]:
plot_data_availability_heatmap(data, id_orders=id_orders)

In [ ]:
data = data.sort([id_, time_]).with_columns(
    target_col.fill_null(target_col.shift(48).over(id_))
)

In [ ]:
plot_missing_percentage(data, id_orders=id_orders)

In [ ]:
plot_data_availability_heatmap(data, id_orders=id_orders)

In [ ]:
data = data.sort([id_, time_]).with_columns(
    target_col.fill_null(target_col.shift(-(48 * 7)).over(id_))
)

In [ ]:
plot_missing_percentage(data, id_orders=id_orders)

In [ ]:
plot_data_availability_heatmap(data, id_orders=id_orders)

In [ ]:
from mlforecast.lag_transforms import (
    RollingStd,
    SeasonalRollingMean,
    SeasonalRollingStd,
    ExponentiallyWeightedMean,
)

lags = [1, 2, 48, 336]
lag_transforms = {
    1: [
        RollingMean(window_size=3),
        RollingMean(window_size=6),
        RollingMean(window_size=12),
        RollingMean(window_size=48),
        RollingStd(window_size=3),
        RollingStd(window_size=6),
        RollingStd(window_size=12),
        RollingStd(window_size=48),
        ExponentiallyWeightedMean(alpha=0.25),
    ],
    48: [
        RollingMean(window_size=7),
        RollingMean(window_size=14),
        RollingStd(window_size=7),
        RollingStd(window_size=14),
        SeasonalRollingMean(season_length=48, window_size=3),
        SeasonalRollingStd(season_length=48, window_size=3),
    ],
    336: [
        RollingMean(window_size=4),
        RollingMean(window_size=8),
        RollingStd(window_size=4),
        RollingStd(window_size=8),
        SeasonalRollingMean(season_length=336, window_size=3),
        SeasonalRollingStd(season_length=336, window_size=3),
    ],
}

In [ ]:
features = [
    partial(
        fourier, season_length=2 * 24, k=10
    ),  # Daily seasonality (48 observations per day)
    partial(
        fourier, season_length=2 * 24 * 7, k=5
    ),  # Weekly seasonality (336 observations per week)
    partial(
        fourier, season_length=2 * 24 * 365, k=3
    ),  # Annual seasonality (approx. 17520 observations per year)
]
data_fourier, data_futr_fourier = pipeline(
    data.select([id_, time_, target_]),
    features=features,
    freq="30m",
    h=48,  # Horizon for future features
)

In [ ]:
sf = StatsForecast(
    models=[
        SklearnModel(
            XGBRegressor(
                n_estimators=100,
                max_depth=6,
                learning_rate=0.1,
                random_state=42,
                tree_method="hist",
            ),
            alias="Local XGB",
        ),
    ],
    freq="30m",
)

y_hat = sf.cross_validation(
    df=mlf.preprocess(data_fourier, static_features=[]),
    h=48,
    step_size=1,
    n_windows=1,
).drop("cutoff")

In [ ]:
metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
mean_eval_df = evaluate(
    y_hat, metrics=metrics, train_df=data.select([id_, time_, target_]), agg_fn="mean"
)
eval_df = evaluate(y_hat, metrics=metrics, train_df=data.select([id_, time_, target_]))

In [ ]:
mlf = MLForecast(
    models=[
        RidgeCV(),
        XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            tree_method="hist",
        ),
    ],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
)

# We'll use cross-validation to evaluate the model's forecasting performance.
y_hat = mlf.cross_validation(
    data_fourier,
    h=48,  # Forecast the next 48 half-hour periods (1 day ahead)
    step_size=1,
    n_windows=1,
    fitted=True,
    static_features=[],
).drop("cutoff")

In [ ]:
mean_eval_df = pl.concat(
    [
        mean_eval_df,
        evaluate(
            y_hat,
            metrics=metrics,
            train_df=data.select([id_, time_, target_]),
            agg_fn="mean",
        ),
    ],
    how="align",
)
eval_df = pl.concat(
    [
        eval_df,
        evaluate(y_hat, metrics=metrics, train_df=data.select([id_, time_, target_])),
    ],
    how="align",
)

In [ ]:
fig = px.box(
    eval_df.unpivot(index=[id_, "metric"], variable_name="model"),
    y="metric",  # Metrics on the x-axis
    x="value",  # Error value on the y-axis
    color="model",  # Models as color
    title="Model Performance Across Metrics",
    labels={"value": "Error Value", "metric": "Metric", "model": "Model"},
)
fig.update_layout(height=600)
fig.show()

In [ ]:
mlf.fit(
    data_fourier,
    static_features=[],
)

In [ ]:
import shap

# SHAP (SHapley Additive exPlanations) helps us understand how each feature contributes to the model's predictions.
# For linear models, SHAP values are equivalent to the feature contributions (coefficient * feature value).

# To compute SHAP values, we need the feature matrix used for training.
# We'll use the MLForecast's internal preprocessed data (after lag/transform creation).
X = (
    mlf.preprocess(data_fourier, static_features=[id_])
    .to_pandas()
    .drop(columns=[id_, time_, target_])
)
fitted_model = mlf.models_[
    "XGBRegressor"
]  # Use the first model (RidgeCV) for SHAP analysis

feature_names = fitted_model.feature_names_in_

# Create a SHAP explainer for the fitted linear regression model
explainer = shap.Explainer(fitted_model, X)

# Compute SHAP values for the first 500 rows (to keep computation and plots manageable)
shap_values = explainer(X.iloc[:500])

# Display a summary plot of SHAP values
shap.summary_plot(shap_values, X.iloc[:500], feature_names=feature_names, show=False)

## When Local Models Outperform Global Models: The Importance of Series Metadata

### Observing the Results

After evaluating both local and global models on our dataset, you may notice that **local models sometimes outperform global models**. This might seem surprising at first, especially since global models have access to much more data and can, in theory, learn broader patterns.

Let's break down why this happens and what it teaches us about modeling time series data.

---

### Why Might Local Models Do Better?

- **Local models** are trained on a single time series at a time. This means they can specialize in the unique patterns, trends, and quirks of each individual series (for example, the specific energy usage habits of one household).
- **Global models**, on the other hand, are trained on *all* series together. If we don't give the global model any information about which series a data point comes from, it tries to learn a "one-size-fits-all" pattern. This can lead to **underfitting**—the model may miss important differences between series.

#### Analogy

Imagine a teacher who tries to teach a class of students using only the average learning style of the group, without knowing anything about each student's strengths or weaknesses. Some students may not get the help they need, and their performance could suffer.

---

### The Missing Ingredient: Series Metadata

To help a global model recognize and adapt to the unique characteristics of each series, we need to provide it with **series metadata**—information that identifies or describes each time series.

#### What is Series Metadata?

- **Series ID:** A unique identifier for each series (e.g., household ID).
- **Static Features:** Attributes that don't change over time, such as building type, region, or customer segment.
- **Groupings:** Categorical variables that group similar series together (e.g., "Acorn" group in our dataset).

#### Why is Metadata Important?

By including metadata, we allow the global model to:

- **Differentiate between series:** The model can learn both shared patterns *and* series-specific behaviors.
- **Leverage similarities:** If two series are similar (e.g., same region or customer type), the model can transfer knowledge between them.
- **Improve accuracy:** The model can make more personalized predictions, reducing errors.

---

### Mathematical Perspective

Without metadata, the global model learns a function:

$$
\hat{y}_{i, t+h} = f(\text{features}_{t}, \theta)
$$

where $\text{features}_{t}$ are only time-based or lagged features, and $\theta$ are shared parameters.

With metadata, the model learns:

$$
\hat{y}_{i, t+h} = f(\text{features}_{i, t}, \text{metadata}_i, \theta)
$$

where $\text{metadata}_i$ provides information about series $i$, allowing the model to adapt its predictions for each series.

---

### Key Takeaway

**If a global model doesn't know which series each observation comes from, it can't personalize its predictions.** This is why, in practice, including series metadata is crucial for global models to match or exceed the performance of local models.

In the next section, we'll explore how to add series metadata to our global models using the `mlforecast` library, and see how this improves forecasting accuracy.

In [ ]:
from lightgbm import LGBMRegressor
from mlforecast.lgb_cv import LightGBMCV

In [ ]:
mlf.fit(
    data_fourier.with_columns(id_col.alias("id").cast(pl.Categorical)).to_pandas(),
    static_features=["id"],
)

In [ ]:
import shap

# SHAP (SHapley Additive exPlanations) helps us understand how each feature contributes to the model's predictions.
# For linear models, SHAP values are equivalent to the feature contributions (coefficient * feature value).

# To compute SHAP values, we need the feature matrix used for training.
# We'll use the MLForecast's internal preprocessed data (after lag/transform creation).
X = mlf.preprocess(
    data_fourier.with_columns(id_col.alias("id").cast(pl.Categorical)).to_pandas(),
    static_features=[id_],
).drop(columns=[id_, time_, target_])

fitted_model = mlf.models_[
    "LGBMRegressor"
]  # Use the first model (RidgeCV) for SHAP analysis

feature_names = fitted_model.feature_names_in_

# Create a SHAP explainer for the fitted linear regression model
# For tree-based models like LGBMRegressor, use shap.TreeExplainer for efficient SHAP value computation.
explainer = shap.TreeExplainer(fitted_model)

# Compute SHAP values for the first 500 rows (to keep computation and plots manageable)
shap_values = explainer(X.iloc[:500])

# Display a summary plot of SHAP values
shap.summary_plot(shap_values, X.iloc[:500], feature_names=feature_names, show=False)

In [ ]:
import plotly.graph_objects as go

# Extract feature names and coefficients from the fitted model
feature_names = fitted_model.feature_names_in_
coefficients = fitted_model.feature_importances_
# Get coefficients and sort features by absolute coefficient value (descending)
coef_df = pl.DataFrame({"feature": feature_names, "coefficient": coefficients}).sort(
    pl.col("coefficient").abs(), descending=True
)
fig = px.bar(
    coef_df,
    x="feature",
    y="coefficient",
    title="Feature Importances",
)

# Add titles and labels for clarity
fig.update_layout(
    title="Feature Importances from Decision Tree Regressor",
    xaxis_title="Feature",
    yaxis_title="Coefficient Value",
    xaxis_tickangle=-45,
    template="plotly_white",
)

fig.show()

In [ ]:
mlf = MLForecast(
    models=[
        LGBMRegressor(
            n_estimators=100,  # Number of boosting rounds (trees)
            max_depth=6,  # Maximum depth of each tree
            learning_rate=0.1,  # Step size shrinkage
            random_state=42,  # For reproducibility
            categorical_feature=["id"],
        )
    ],
    freq="30min",
    lags=lags,
    lag_transforms=lag_transforms,
)

# We'll use cross-validation to evaluate the model's forecasting performance.
y_hat = pl.from_pandas(
    mlf.cross_validation(
        data_fourier.with_columns(id_col.alias("id").cast(pl.Categorical)).to_pandas(),
        h=48,  # Forecast the next 48 half-hour periods (1 day ahead)
        step_size=1,
        n_windows=1,
        fitted=True,
        static_features=["id"],
    )
).drop("cutoff")

In [ ]:
mlf = LightGBMCV(
    models=[
        LGBMRegressor(
            n_estimators=100,  # Number of boosting rounds (trees)
            max_depth=6,  # Maximum depth of each tree
            learning_rate=0.1,  # Step size shrinkage
            random_state=42,  # For reproducibility
            categorical_feature=["id"],
        )
    ],
    freq="30min",
    lags=lags,
    lag_transforms=lag_transforms,
)

# We'll use cross-validation to evaluate the model's forecasting performance.
y_hat = pl.from_pandas(
    mlf.cross_validation(
        data_fourier.with_columns(id_col.alias("id").cast(pl.Categorical)).to_pandas(),
        h=48,  # Forecast the next 48 half-hour periods (1 day ahead)
        step_size=1,
        n_windows=1,
        fitted=True,
        static_features=["id"],
    )
).drop("cutoff")

In [ ]:
mean_eval_df = pl.concat(
    [
        mean_eval_df,
        evaluate(
            y_hat,
            metrics=metrics,
            train_df=data.select([id_, time_, target_]),
            agg_fn="mean",
        ),
    ],
    how="align",
)
eval_df = pl.concat(
    [
        eval_df,
        evaluate(y_hat, metrics=metrics, train_df=data.select([id_, time_, target_])),
    ],
    how="align",
)

In [ ]:
fig = px.box(
    eval_df.unpivot(index=[id_, "metric"], variable_name="model"),
    y="metric",  # Metrics on the x-axis
    x="value",  # Error value on the y-axis
    color="model",  # Models as color
    title="Model Performance Across Metrics",
    labels={"value": "Error Value", "metric": "Metric", "model": "Model"},
)
fig.update_layout(height=600)
fig.show()